## 5. Simulación de un Registro de Logs:

o Crea una lista enlazada simple para almacenar registros de logs de un
sistema (fecha, hora, mensaje).

o Implementa un método para buscar registros que coincidan con un
determinado mensaje de error.


o Crea una función que elimine automáticamente los registros de más de 30
días de antigüedad.

In [4]:
import json
from datetime import datetime, timedelta
from Clases_estructuras import lista_enlazada_simple

ruta = "Datos/Logs.json"

def cargar_logs_en_lista(ruta_archivo: str) -> lista_enlazada_simple:
    """
    Carga los registros del archivo JSON en una lista enlazada simple.

    Parametros:
    - ruta_archivo (str): Ruta del archivo JSON con la lista de logs.

    Retorna:
    - lista_enlazada_simple: Lista enlazada con todos los registros.
    """
    lista = lista_enlazada_simple()
    with open(ruta_archivo, "r", encoding="utf-8") as file:
        data = json.load(file)
        for item in data:
            lista.agregar(item)
    return lista

def buscar_registro(lista: lista_enlazada_simple, error: int) -> list:
    """
    Busca todos los registros cuyo Tipo_error coincida con el valor indicado.

    Logica:
    - Recorre nodo por nodo la lista enlazada.
    - Compara el campo 'Tipo_error' de cada registro con 'error'.
    - Acumula y retorna las coincidencias.

    Parametros:
    - lista (lista_enlazada_simple): Lista de logs.
    - error (int): Codigo de error a buscar.

    Retorna:
    - list: Lista de diccionarios que coinciden con el error.
    """
    coincidencias = []
    nodo = lista.head

    while nodo is not None:
        if nodo.data.get("Tipo_error") == error:
            coincidencias.append(nodo.data)
        nodo = nodo.next

    return coincidencias

def comparar_fechas(fecha_registro: str, dia: int, mes: int, año: int) -> bool:
    """
    Determina si un registro tiene mas de 30 dias de antiguedad
    respecto a una fecha base.

    Esto funciona de la siguiente manera:
    - Convierte la fecha del registro (formato dd-mm-YYYY) a datetime.
    - Crea la fecha base con dia, mes y año.
    - Calcula la fecha limite = fecha_base - 30 dias.
    - Si fecha_registro < fecha_limite, el registro es antiguo (True).

    tiene parametros:
    - fecha_registro (str): Fecha del log en formato 'dd-mm-YYYY'.
    - dia (int): Dia de la fecha base.
    - mes (int): Mes de la fecha base.
    - año (int): Año de la fecha base.

    Retorna:
    - bool: True si tiene mas de 30 dias de antiguedad, False si no.
    """
    fecha_registro_dt = datetime.strptime(fecha_registro, "%d-%m-%Y")
    fecha_base = datetime(año, mes, dia)
    fecha_limite = fecha_base - timedelta(days=30)
    return fecha_registro_dt < fecha_limite

def eliminar_por_antiguedad(lista: lista_enlazada_simple, dia: int, mes: int, año: int) -> int:
    """
    Elimina automaticamente de la lista los registros con mas de 30 dias
    de antiguedad respecto a la fecha base indicada.

    Logica de eliminacion en lista enlazada simple:
    - Recorre la lista con dos punteros: 'anterior' y 'actual'.
    - Si el nodo actual debe eliminarse:
    - Si es la cabeza, mueve head al siguiente nodo.
    - Si es intermedio/final, conecta anterior.next con actual.next.
    - Actualiza tail al final para mantener la estructura consistente.

    Parametros:
    - lista (lista_enlazada_simple): Lista de logs.
    - dia (int): Dia de la fecha base.
    - mes (int): Mes de la fecha base.
    - año (int): Año de la fecha base.

    Retorna:
    - int: Cantidad de registros eliminados.
    """
    actual = lista.head
    anterior = None
    eliminados = 0

    while actual is not None:
        fecha_log = actual.data.get("fecha")
        debe_eliminarse = comparar_fechas(fecha_log, dia, mes, año)

        if debe_eliminarse:
            eliminados += 1
            if anterior is None:
                lista.head = actual.next
                actual = lista.head
            else:
                anterior.next = actual.next
                actual = actual.next
        else:
            anterior = actual
            actual = actual.next

    # Recalcular tail para evitar referencias invalidas tras eliminaciones.
    if lista.head is None:
        lista.tail = None
    else:
        nodo = lista.head
        while nodo.next is not None:
            nodo = nodo.next
        lista.tail = nodo

    return eliminados


# ----------------------------
# EJEMPLO DE USO
# ----------------------------
lista = cargar_logs_en_lista(ruta)

# 1) Buscar registros por codigo de error
error_a_buscar = 322
resultado = buscar_registro(lista, error_a_buscar)
print(f"Coincidencias para el error {error_a_buscar}: {len(resultado)}")
for registro in resultado:
    print(registro)

# 2) Eliminar registros con mas de 30 dias de antiguedad
# Fecha base de ejemplo: 28/04/2026
eliminados = eliminar_por_antiguedad(lista, 28, 4, 2026)
print(f"Registros eliminados por antiguedad (>30 dias): {eliminados}")

lista.mostrar()

Coincidencias para el error 322: 2
{'fecha': '12-03-2026', 'hora': '4:39 AM', 'Tipo_error': 322}
{'fecha': '03-07-2025', 'hora': '9:31 PM', 'Tipo_error': 322}
Registros eliminados por antiguedad (>30 dias): 97
{'fecha': '25-04-2026', 'hora': '3:28 AM', 'Tipo_error': 340} -> {'fecha': '17-04-2026', 'hora': '1:54 PM', 'Tipo_error': 356} -> {'fecha': '11-04-2026', 'hora': '2:37 AM', 'Tipo_error': 262} -> None


```pseudocodigo
funcion cargar_logs_en_lista(ruta_archivo)
 lista = nueva lista_enlazada_simple()
 abrir archivo ruta_archivo en modo lectura
 data = leer JSON del archivo
 Para cada item en data
  lista.agregar(item)
 Cerrar archivo
 Retornar lista

funcion buscar_registro(lista, error)
 coincidencias = []
 nodo = lista.head
 Mientras nodo != Nulo
  Si nodo.data.get("Tipo_error") == error
   coincidencias.anadir(nodo.data)
  nodo = nodo.next
 Retornar coincidencias

funcion comparar_fechas(fecha_registro, dia, mes, año)
 fecha_registro_dt = convertir fecha_registro formato "%d-%m-%Y" a fecha
 fecha_base = fecha(año, mes, dia)
 fecha_limite = fecha_base - 30 dias
 Retornar fecha_registro_dt < fecha_limite

funcion eliminar_por_antiguedad(lista, dia, mes, año)
 actual = lista.head
 anterior = Nulo
 eliminados = 0
 Mientras actual != Nulo
  fecha_log = actual.data.get("fecha")
  debe_eliminarse = comparar_fechas(fecha_log, dia, mes, año)
  Si debe_eliminarse
   eliminados = eliminados + 1
   Si anterior es Nulo
    lista.head = actual.next
    actual = lista.head
   Sino
    anterior.next = actual.next
    actual = actual.next
  Sino
   anterior = actual
   actual = actual.next
 Si lista.head es Nulo
  lista.tail = Nulo
 Sino
  nodo = lista.head
  Mientras nodo.next != Nulo
   nodo = nodo.next
  lista.tail = nodo
 Retornar eliminados

# EJEMPLO DE USO
lista = cargar_logs_en_lista(ruta)
resultado = buscar_registro(lista, 322)
Imprimir "Coincidencias para el error 322:" + longitud(resultado)
eliminados = eliminar_por_antiguedad(lista, 28, 4, 2026)
Imprimir "Registros eliminados por antiguedad (>30 dias):" + eliminados
lista.mostrar()
```